# Oppgave 1: Modellering av et meslingutbrudd

## Oppgave 1a: Markovkjede og overgangsmatrise
Prosessen $\{X_n\}$ er en Markovkjede fordi tilstanden til personen neste dag bare avhenger av hvilken tilstand personen er i i dag, og ikke av tidligere tilstander.

Vi har tre tilstander:

- $S$: mottakelig for smitte
- $I$: smittet
- $R$: frisk og immun

Disse tilsvarer henholdsvis tilstandene $0$, $1$ og $2$.

Hvis personen er i $S$, blir personen smittet neste dag med sannsynlighet $\beta$, og forblir i $S$ med sannsynlighet $1-\beta$.

Hvis personen er i $I$, blir personen frisk og immun med sannsynlighet $\gamma$, og forblir smittet med sannsynlighet $1-\gamma$.

Hvis personen er i $R$, mister personen immuniteten og går tilbake til $S$ med sannsynlighet $\alpha$, og forblir i $R$ med sannsynlighet $1-\alpha$.

Dermed blir overgangsmatrisen

$$
P =
\begin{pmatrix}
1-\beta & \beta & 0 \\
0 & 1-\gamma & \gamma \\
\alpha & 0 & 1-\alpha
\end{pmatrix}.
$$

Her er $\beta$ sannsynligheten for å gå fra $S$ til $I$, $\gamma$ sannsynligheten for å gå fra $I$ til $R$, og $\alpha$ sannsynligheten for å gå fra $R$ til $S$.

## Oppgave 1b: Grensefordeling og forventet antall dager



For $\beta=0.01$, $\gamma=0.10$ og $\alpha=0.005$ får vi overgangsmatrisen

$$
P=
\begin{pmatrix}
0.99 & 0.01 & 0 \\
0 & 0.90 & 0.10 \\
0.005 & 0 & 0.995
\end{pmatrix}.
$$

Markovkjeden er irreducibel siden det er mulig å komme fra enhver tilstand til enhver annen tilstand gjennom kjeden

$$
S \to I \to R \to S.
$$

Den er også aperiodisk, siden hver tilstand har positiv sannsynlighet for å bli værende i samme tilstand. Dermed har den en unik limiting distribution.

La den stasjonære fordelingen være

$$
\pi=(\pi_S,\pi_I,\pi_R).
$$

Vi løser

$$
\pi P=\pi
$$

sammen med

$$
\pi_S+\pi_I+\pi_R=1.
$$

Fra ligningene får vi

$$
0.01\pi_S=0.10\pi_I
$$

slik at

$$
\pi_I=0.1\pi_S.
$$

Videre får vi

$$
0.10\pi_I=0.005\pi_R,
$$

og dermed

$$
\pi_R=20\pi_I=2\pi_S.
$$

Ved å bruke

$$
\pi_S+\pi_I+\pi_R=1
$$

får vi

$$
\pi_S+0.1\pi_S+2\pi_S=1,
$$

så

$$
\pi_S=\frac{1}{3.1}\approx0.3226.
$$

Dermed blir

$$
\pi_I\approx0.0323,
\qquad
\pi_R\approx0.6452.
$$

Den langsiktige andelen tid i hver tilstand er derfor omtrent

$$
(\pi_S,\pi_I,\pi_R)
=
(0.3226,0.0323,0.6452).
$$

Siden ett år har 365 dager, blir det gjennomsnittlige antallet dager per år

$$
365\pi_S\approx117.7,
$$

$$
365\pi_I\approx11.8,
$$

$$
365\pi_R\approx235.5.
$$

Altså vil personen i det lange løp i gjennomsnitt være omtrent 118 dager mottakelig, 12 dager smittet og 235 dager frisk og immun per år.

## Oppgave 1c: Simulering og konfidensintervaller

In [ ]:
import numpy as np
from scipy.stats import t


P = np.array([
    [0.99, 0.01, 0.00],   # S
    [0.00, 0.90, 0.10],   # I
    [0.005, 0.00, 0.995]   # R
])

n_steps = 7300      
burn_in = 3650   
n_runs = 30          

def simulate_chain():
   
    x = np.zeros(n_steps + 1, dtype=int)

    for n in range(n_steps):
        current_state = x[n]

        x[n + 1] = np.random.choice(
            [0, 1, 2],
            p=P[current_state]
        )

    return x



estimates = np.zeros((n_runs, 3))

for run in range(n_runs):

    x = simulate_chain()

    # Bruk bare de siste 10 årene
    last_10_years = x[-3650:]

    for state in [0, 1, 2]:

        proportion = np.mean(last_10_years == state)

        # Gjør om andel av tiden til dager per år
        estimates[run, state] = proportion * 365


print("Estimater fra de 30 simuleringene:")
print(estimates)

means = estimates.mean(axis=0)
standard_errors = estimates.std(axis=0, ddof=1) / np.sqrt(n_runs)
critical_value = t.ppf(0.975, df=n_runs - 1)
lower = means - critical_value * standard_errors
upper = means + critical_value * standard_errors

state_names = ["S", "I", "R"]
print("\nGjennomsnitt og omtrentlige 95 % konfidensintervaller:")
for state, mean, low, high in zip(state_names, means, lower, upper):
    print(f"{state}: {mean:.1f} dager per år, 95 % KI [{low:.1f}, {high:.1f}]")

Estimater fra de 30 simuleringene:
[[ 83.6   7.3 274.1]
 [120.7  11.1 233.2]
 [154.2   7.6 203.2]
 [164.2   8.1 192.7]
 [ 61.7   6.7 296.6]
 [108.6  10.  246.4]
 [160.3   7.2 197.5]
 [147.8  10.3 206.9]
 [ 43.2   7.3 314.5]
 [105.8   8.8 250.4]
 [ 72.    4.1 288.9]
 [ 80.8   9.6 274.6]
 [145.1   7.3 212.6]
 [115.5  15.5 234. ]
 [ 44.4   2.7 317.9]
 [115.   18.9 231.1]
 [141.3  14.8 208.9]
 [164.8   7.1 193.1]
 [136.2  13.2 215.6]
 [132.1  19.  213.9]
 [ 98.2  12.5 254.3]
 [100.5  15.2 249.3]
 [132.4  15.5 217.1]
 [ 74.3  15.8 274.9]
 [ 85.   10.7 269.3]
 [ 58.3   7.9 298.8]
 [121.5   8.3 235.2]
 [ 95.4  12.  257.6]
 [ 72.3   8.8 283.9]
 [123.9   6.4 234.7]]

Gjennomsnitt og omtrentlige 95 % konfidensintervaller:
S: 108.6 dager per år, 95 % KI [95.3, 121.9]
I: 10.3 dager per år, 95 % KI [8.8, 11.9]
R: 246.0 dager per år, 95 % KI [232.3, 259.8]


### Metode og resultater

Markovkjeden ble simulert i 7300 tidssteg, tilsvarende 20 år. 
Kjeden startet i tilstand $S$. For hver simulering brukte vi kun de siste 
3650 tidsstegene, tilsvarende de siste 10 årene, til å estimere den 
langsiktige andelen tid i tilstandene $S$, $I$ og $R$.

For hver tilstand beregnet vi estimert antall dager per år ved

$$
\hat d_j = 365\frac{N_j}{3650},
$$

der $N_j$ er antall dager tilbrakt i tilstand $j$ i løpet av de siste 
10 årene.

Denne simuleringen ble gjentatt 30 uavhengige ganger. For hver tilstand 
fikk vi dermed 30 estimater. Vi beregnet gjennomsnittet $\bar d_j$ og 
utvalgsstandardavviket $s_j$, og brukte et tilnærmet 95\% 
konfidensintervall

$$
\bar d_j \pm t_{0.975,29}\frac{s_j}{\sqrt{30}}.
$$

Den aktuelle kjøringen ga følgende resultater:

| Tilstand | Estimert gjennomsnitt (dager per år) | 95\% konfidensintervall | Eksakt verdi fra 1b |
|:--:|--:|:--:|--:|
| $S$ | 108.6 | $[95.3, 121.9]$ | 117.7 |
| $I$ | 10.3 | $[8.8, 11.9]$ | 11.8 |
| $R$ | 246.0 | $[232.3, 259.8]$ | 235.5 |

De eksakte verdiene fra 1b ligger innenfor de respektive 95\% konfidensintervallene. Resultatene fra simuleringen er derfor kompatible med håndberegningene. At de estimerte gjennomsnittene ikke er helt like de eksakte verdiene, er forventet fordi simuleringene har endelig lengde og dermed inneholder tilfeldig variasjon. Konfidensintervallene uttrykker usikkerheten i de estimerte gjennomsnittene.

## Oppgave 1d: Markovegenskap

Skriv besvarelsen her.

## Oppgave 1e: Simulering av populasjonsmodellen

## Oppgave 1f: Maksimalt antall smittede

## Oppgave 1g: Vaksinasjon